# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
# ML-04 — Load FlyRank Warehouse

import duckdb
import os
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(f"""
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# FlyRank warehouse
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test access to the main fact table
result = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│   78835655 │
└────────────┘



In [4]:
# Inspect the warehouse schema

schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 1
""")

print(schema)

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# Section 1 — Unit of analysis + time window

print("""
Lane: Refresh / Content Opportunity Scoring

Unit of analysis:
One row represents one content item for one client on one report date.

Table:
fact_content_daily_performance

Time window:
Use a mid-panel month, March 2026 (month = '2026-03'),
for development and verification.

Prediction / ranking target:
Rank content items by their opportunity for refresh, using
historical performance signals such as impressions, sessions,
engagement, and trend direction.

Deliberate exclusion:
Exclude any future-period outcome or label-derived field that
would only be known after the refresh decision moment.
""")


Lane: Refresh / Content Opportunity Scoring

Unit of analysis:
One row represents one content item for one client on one report date.

Table:
fact_content_daily_performance

Time window:
Use a mid-panel month, March 2026 (month = '2026-03'),
for development and verification.

Prediction / ranking target:
Rank content items by their opportunity for refresh, using
historical performance signals such as impressions, sessions,
engagement, and trend direction.

Deliberate exclusion:
Exclude any future-period outcome or label-derived field that
would only be known after the refresh decision moment.



In [6]:
# Verify the proposed unit of analysis

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT report_date || '|' || content_hash_id || '|' || client_hash_id) AS unique_row_keys
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────────────┐
│  rows   │ unique_row_keys │
│  int64  │      int64      │
├─────────┼─────────────────┤
│ 9841378 │         9841378 │
└─────────┴─────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
print("""
FEATURES:
- impressions_90d
- sessions_90d
- engagement_rate
- trend_direction
- ctr

LABEL / PROXY:
- refresh_opportunity (derived later from future performance)

CONTEXT:
- report_date
- month
- content_type
- main_intent
- competition_level
- client_hash_id

EXCLUDED:
- Future-period performance outcomes
- Any label-derived fields
- Fields that would only be known after the refresh decision
- Client-identifying information
""")


FEATURES:
- impressions_90d
- sessions_90d
- engagement_rate
- trend_direction
- ctr

LABEL / PROXY:
- refresh_opportunity (derived later from future performance)

CONTEXT:
- report_date
- month
- content_type
- main_intent
- competition_level
- client_hash_id

EXCLUDED:
- Future-period performance outcomes
- Any label-derived fields
- Fields that would only be known after the refresh decision
- Client-identifying information



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# QUERY 1 — Verify the grain
grain_query = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || content_hash_id || '|' || client_hash_id) AS unique_grain_keys
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

print("QUERY 1 — GRAIN")
print(grain_query)


# QUERY 2 — Verify row count and date window
window_query = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

print("\nQUERY 2 — ROW COUNT AND DATE SPAN")
print(window_query)


# QUERY 3 — Verify data availability

availability_query = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

print("QUERY 3 — AVAILABILITY")
print(availability_query)

QUERY 1 — GRAIN


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_keys │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘


QUERY 2 — ROW COUNT AND DATE SPAN
┌────────────┬────────────┬────────────┐
│ march_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

QUERY 3 — AVAILABILITY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
print("""
DATA LIMITATIONS

1. The dataset does not provide a complete view of every historical
   period for every content item, so comparisons across time can be
   unbalanced.

2. Google Search Console (GSC) and Google Analytics 4 (GA4)
   availability varies by row. Therefore, features based on these
   sources cannot be assumed to exist for every content item.

3. The 90-day performance fields represent fixed reporting windows,
   so overlapping windows may make observations across nearby dates
   correlated.

4. The dataset is anonymized. Client and content identifiers are
   pseudonymized, so the data cannot be used to identify real clients
   or content owners.

5. June 2026 is treated as a sealed final month and is not used for
   development or label design.
""")


DATA LIMITATIONS

1. The dataset does not provide a complete view of every historical
   period for every content item, so comparisons across time can be
   unbalanced.

2. Google Search Console (GSC) and Google Analytics 4 (GA4)
   availability varies by row. Therefore, features based on these
   sources cannot be assumed to exist for every content item.

3. The 90-day performance fields represent fixed reporting windows,
   so overlapping windows may make observations across nearby dates
   correlated.

4. The dataset is anonymized. Client and content identifiers are
   pseudonymized, so the data cannot be used to identify real clients
   or content owners.

5. June 2026 is treated as a sealed final month and is not used for
   development or label design.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.